In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import json

# synthetic training data for OSF's creative services
np.random.seed(42)

services = ['Logo Design', 'Web Design', 'Editing', 'Ads', 'Branding']
n_samples = 200

# dataset with price ranges for each service
data = []
for _ in range(n_samples):
    service = np.random.choice(services)

    # Base prices and variations for each service type
    if service == 'Logo Design':
        base = 500
        variation = np.random.normal(0, 150)
        complexity_factor = np.random.uniform(0.8, 1.5)
    elif service == 'Web Design':
        base = 2500
        variation = np.random.normal(0, 800)
        complexity_factor = np.random.uniform(0.7, 2.0)
    elif service == 'Editing':
        base = 300
        variation = np.random.normal(0, 100)
        complexity_factor = np.random.uniform(0.9, 1.4)
    elif service == 'Ads':
        base = 800
        variation = np.random.normal(0, 250)
        complexity_factor = np.random.uniform(0.8, 1.6)
    else:  # Branding
        base = 3500
        variation = np.random.normal(0, 1000)
        complexity_factor = np.random.uniform(0.8, 1.8)

    cost = max(100, base * complexity_factor + variation)
    data.append({'service': service, 'cost': round(cost, 2)})

df = pd.DataFrame(data)

# Prepare features and target
le = LabelEncoder()
X = le.fit_transform(df['service']).reshape(-1, 1)
y = df['cost'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest model
model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 60)
print("OSF CREATIVE SERVICES - COST PREDICTION MODEL")
print("=" * 60)
print(f"\nModel Performance:")
print(f"  Mean Absolute Error: ${mae:.2f}")
print(f"  R² Score: {r2:.3f}")
print()

# Function to predict cost for a service
def predict_service_cost(service_type):
    """
    Predict estimated cost range for a given service type.

    Args:
        service_type (str): One of the OSF services

    Returns:
        dict: JSON with estimated cost range
    """
    if service_type not in services:
        return {
            "error": f"Invalid service. Choose from: {', '.join(services)}"
        }

    # Encode service
    service_encoded = le.transform([service_type]).reshape(-1, 1)

    # Get predictions from all trees for confidence interval
    predictions = np.array([tree.predict(service_encoded)[0] for tree in model.estimators_])

    mean_cost = predictions.mean()
    std_cost = predictions.std()

    # Calculate range (mean ± 1 std deviation covers ~68% of cases)
    min_cost = max(100, mean_cost - std_cost)
    max_cost = mean_cost + std_cost
    usd_to_inr = 83

    result = {
        "service": service_type,
        "estimated_cost": {
            "min": round(min_cost * usd_to_inr, 2),
            "max": round(max_cost * usd_to_inr, 2),
            "average": round(mean_cost * usd_to_inr, 2)
        },
        "currency": "INR",
        "note": "Actual cost may vary based on project complexity and requirements"
    }

    return result

# predictions for all services
print("=" * 60)
print("PREDICTIONS FOR ALL SERVICES")
print("=" * 60)
print()

for service in services:
    prediction = predict_service_cost(service)
    print(f"Service: {prediction['service']}")
    print(f"  Estimated Range: Rs.{prediction['estimated_cost']['min']:.2f} - Rs.{prediction['estimated_cost']['max']:.2f}")
    print(f"  Average: Rs.{prediction['estimated_cost']['average']:.2f}")
    print()

# JSON output
print("=" * 60)
print("JSON prediction OUTPUT")
print("=" * 60)
print()

print("1: Web Design")
print(json.dumps(predict_service_cost("Web Design"), indent=2))
print()

print("2: Logo Design")
print(json.dumps(predict_service_cost("Logo Design"), indent=2))
print()

print("3: Branding")
print(json.dumps(predict_service_cost("Branding"), indent=2))
print()

print("4: Editing")
print(json.dumps(predict_service_cost("Editing"), indent=2))
print()

print("5: Ads")
print(json.dumps(predict_service_cost("Ads"), indent=2))
print()

# Interactive user input for custom prediction
print("=" * 60)
print("PREDICTION")
print("=" * 60)
print(f"\nAvailable services: {', '.join(services)}")
print()

test_service = input("Enter service name to predict cost: ").strip()
result = predict_service_cost(test_service)

if "error" in result:
    print(f"\nError: {result['error']}")
else:
    print(f"\nPredicting cost for: {test_service}")
    print(json.dumps(result, indent=2))

    # Save prediction to JSON file
    output_filename = f"osf_cost_prediction_{test_service.replace(' ', '_').lower()}.json"
    with open(output_filename, 'w') as f:
        json.dump(result, f, indent=2)
    print(f"\nPrediction saved to: {output_filename}")
print()

# Service statistics
print("=" * 60)
print("TRAINING DATA STATISTICS")
print("=" * 60)
print()
stats = df.groupby('service')['cost'].agg(['mean', 'min', 'max', 'std'])
stats.columns = ['Average', 'Min', 'Max', 'Std Dev']
print(stats.round(2))
print()

# Save all predictions to a JSON file
all_predictions = {
    "model_info": {
        "model_type": "Random Forest Regressor",
        "currency": "INR",
        "exchange_rate": "1 USD = 83 INR",
        "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    },
    "predictions": {}
}

for service in services:
    all_predictions["predictions"][service] = predict_service_cost(service)

# Save comprehensive predictions
with open("osf_all_services_predictions.json", 'w') as f:
    json.dump(all_predictions, f, indent=2)

print("=" * 60)
print("ALL PREDICTIONS SAVED")
print("=" * 60)
print("File: osf_all_services_predictions.json")
print()

print("=" * 60)
print("MODEL READY FOR PREDICTIONS!")
print("=" * 60)
print("\nUsage: predict_service_cost('Service Name')")
print(f"Available services: {', '.join(services)}")

OSF CREATIVE SERVICES - COST PREDICTION MODEL

Model Performance:
  Mean Absolute Error: $611.23
  R² Score: 0.772

PREDICTIONS FOR ALL SERVICES

Service: Logo Design
  Estimated Range: Rs.41531.02 - Rs.46907.77
  Average: Rs.44219.40

Service: Web Design
  Estimated Range: Rs.246250.62 - Rs.277628.52
  Average: Rs.261939.57

Service: Editing
  Estimated Range: Rs.29017.44 - Rs.30937.52
  Average: Rs.29977.48

Service: Ads
  Estimated Range: Rs.75273.85 - Rs.84307.59
  Average: Rs.79790.72

Service: Branding
  Estimated Range: Rs.372805.05 - Rs.417929.83
  Average: Rs.395367.44

JSON prediction OUTPUT

1: Web Design
{
  "service": "Web Design",
  "estimated_cost": {
    "min": 246250.62,
    "max": 277628.52,
    "average": 261939.57
  },
  "currency": "INR",
  "note": "Actual cost may vary based on project complexity and requirements"
}

2: Logo Design
{
  "service": "Logo Design",
  "estimated_cost": {
    "min": 41531.02,
    "max": 46907.77,
    "average": 44219.4
  },
  "currency"